# Nigeria House Price — EDA
Exploring the dataset used to train the price predictor. Covers Abuja and Port Harcourt listings only.

In [ ]:
import sys
sys.path.append('../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import os

from preprocess import load_and_clean, CITY_AREAS

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

DATA_PATH = '../backend/data/nigeria_houses.csv'
MODEL_DIR = '../backend/saved_model'

## 1. Raw dataset

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f'Shape: {raw.shape}')
print(f'Columns: {raw.columns.tolist()}')
raw.head()

In [ ]:
print('Missing values:')
print(raw.isnull().sum())
print()
print('States in the dataset:')
print(raw['state'].value_counts())

## 2. After cleaning (Abuja + Port Harcourt only)

In [ ]:
df = load_and_clean(DATA_PATH)
print(f'Rows after cleaning: {len(df)}')
print()
counts = df['city'].value_counts()
print(counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
counts.plot(kind='bar', ax=ax, color=['#16a34a', '#2563eb'], edgecolor='white', width=0.5)
ax.set_title('Listings per city', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Price distribution

In [ ]:
print('Price statistics (₦):')
stats = df['price'].describe()
for label, val in stats.items():
    print(f'  {label:5s}  ₦{val:>18,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# raw price histogram
axes[0].hist(df['price'] / 1e6, bins=60, color='#16a34a', edgecolor='white', alpha=0.85)
axes[0].set_title('Price distribution (₦M)', fontweight='bold')
axes[0].set_xlabel('Price (₦ millions)')
axes[0].set_ylabel('Count')

# log-transformed (what the model actually sees)
axes[1].hist(np.log1p(df['price']), bins=60, color='#2563eb', edgecolor='white', alpha=0.85)
axes[1].set_title('Log(price) — model target', fontweight='bold')
axes[1].set_xlabel('log1p(price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()
print('The log transform makes the skewed distribution much more normal — that is why the model trains on it.')

## 4. Price by city

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
city_order = ['Abuja', 'Port Harcourt']
palette = {'Abuja': '#16a34a', 'Port Harcourt': '#2563eb'}

sns.boxplot(
    data=df, x='city', y='price', order=city_order,
    palette=palette, width=0.45, fliersize=2, linewidth=1.2, ax=ax
)
ax.set_title('Price distribution by city', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Price (₦)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₦{x/1e6:.0f}M'))

# annotate medians
for i, city in enumerate(city_order):
    med = df[df['city'] == city]['price'].median()
    ax.text(i, med, f'  ₦{med/1e6:.0f}M', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Property types

In [ ]:
type_stats = (
    df.groupby('property_type')['price']
    .agg(count='count', median='median')
    .sort_values('median', ascending=False)
)
print(type_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# listing count
type_count = df['property_type'].value_counts()
axes[0].barh(type_count.index, type_count.values, color='#16a34a', edgecolor='white')
axes[0].set_title('Listings per property type', fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# median price
med_price = df.groupby('property_type')['price'].median().sort_values(ascending=True)
axes[1].barh(med_price.index, med_price.values / 1e6, color='#2563eb', edgecolor='white')
axes[1].set_title('Median price per property type (₦M)', fontweight='bold')
axes[1].set_xlabel('Median price (₦M)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Bedroom & bathroom distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {'Abuja': '#16a34a', 'Port Harcourt': '#2563eb'}

for city, grp in df.groupby('city'):
    axes[0].hist(grp['bedrooms'], bins=range(1, 12), alpha=0.65,
                 label=city, color=colors[city], edgecolor='white', density=True)
    axes[1].hist(grp['bathrooms'], bins=range(1, 12), alpha=0.65,
                 label=city, color=colors[city], edgecolor='white', density=True)

axes[0].set_title('Bedroom count', fontweight='bold')
axes[0].set_xlabel('Bedrooms')
axes[0].set_ylabel('Density')
axes[0].legend()

axes[1].set_title('Bathroom count', fontweight='bold')
axes[1].set_xlabel('Bathrooms')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Top areas by median price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, city, color in zip(axes, ['Abuja', 'Port Harcourt'], ['#16a34a', '#2563eb']):
    subset = df[df['city'] == city]
    # only areas with at least 5 listings to avoid noise
    area_med = (
        subset.groupby('town')['price']
        .agg(count='count', median='median')
        .query('count >= 5')
        .sort_values('median', ascending=False)
        .head(15)
    )
    ax.barh(area_med.index[::-1], area_med['median'][::-1] / 1e6, color=color, edgecolor='white')
    ax.set_title(f'{city} — top 15 areas by median price', fontweight='bold')
    ax.set_xlabel('Median price (₦M)')

plt.tight_layout()
plt.show()

## 8. Feature correlations with price

In [ ]:
num_cols = ['bedrooms', 'bathrooms', 'toilets', 'parking_space', 'price']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax, square=True
)
ax.set_title('Feature correlation matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# price vs bedrooms scatter
fig, ax = plt.subplots(figsize=(7, 4))
palette = {'Abuja': '#16a34a', 'Port Harcourt': '#2563eb'}

for city, grp in df.groupby('city'):
    ax.scatter(grp['bedrooms'], grp['price'] / 1e6, alpha=0.25, s=15,
               color=palette[city], label=city)

ax.set_xlabel('Bedrooms')
ax.set_ylabel('Price (₦M)')
ax.set_title('Price vs bedrooms', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Model feature importances

In [ ]:
model = joblib.load(os.path.join(MODEL_DIR, 'model.pkl'))
feature_cols = joblib.load(os.path.join(MODEL_DIR, 'feature_cols.pkl'))

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind='barh', ax=ax, color='#1d6f42', edgecolor='white')
ax.set_title('XGBoost feature importances', fontweight='bold')
ax.set_xlabel('Importance score')
for p in ax.patches:
    ax.annotate(f'{p.get_width():.3f}', (p.get_width(), p.get_y() + p.get_height() / 2),
                va='center', ha='left', fontsize=9, color='#374151')
plt.tight_layout()
plt.show()

## 10. Price multiplier validation
Cross-check that our hand-crafted area multipliers roughly match actual median prices in the data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, city, color in zip(axes, ['Abuja', 'Port Harcourt'], ['#16a34a', '#2563eb']):
    subset = df[df['city'] == city]
    city_median = subset['price'].median()

    area_data = []
    for area, multiplier in CITY_AREAS[city].items():
        rows = subset[subset['town'].str.lower() == area.lower()]
        if len(rows) >= 3:
            actual_ratio = rows['price'].median() / city_median
            area_data.append({'area': area, 'actual': actual_ratio, 'model': multiplier})

    if not area_data:
        ax.set_title(f'{city} — not enough data per area')
        continue

    cmp = pd.DataFrame(area_data).set_index('area').sort_values('actual', ascending=False)
    x = range(len(cmp))
    ax.bar([i - 0.2 for i in x], cmp['actual'], width=0.38, label='Actual ratio', color=color, alpha=0.8)
    ax.bar([i + 0.2 for i in x], cmp['model'], width=0.38, label='Model multiplier', color='#f59e0b', alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(cmp.index, rotation=45, ha='right', fontsize=7)
    ax.axhline(1.0, color='#94a3b8', linestyle='--', linewidth=0.8)
    ax.set_title(f'{city} — actual vs model area multipliers', fontweight='bold')
    ax.set_ylabel('Multiplier vs city median')
    ax.legend()

plt.tight_layout()
plt.show()
print('Areas with few listings have noisier actual ratios — the model multipliers are deliberately smoothed estimates.')